### DuPont 3-Factor: Key Observations

The tree diagram above makes the multiplicative structure of ROE concrete.  Notice how the same line items (Revenue, Total Assets) appear in *multiple* branches — this is not redundancy but reflects the algebraic cancellation that makes the identity work.

> **Key Concept:** When Revenue appears in both the numerator of Asset Turnover and the denominator of Net Profit Margin, the two Revenue terms cancel, leaving NI/Assets.  Multiply by Assets/Equity and you recover NI/Equity = ROE.  The decomposition is an **identity**, not an approximation — it holds exactly for every firm in every period.

# Financial Ratio Analysis & DuPont Decomposition

*CFA Level 1 — Financial Statement Analysis*

This notebook develops a **from-scratch** framework for evaluating company performance through financial ratios and the DuPont decomposition of Return on Equity. We construct all metrics from raw financial-statement data using only NumPy, derive every formula algebraically, and visualise the results with Matplotlib.

**Learning objectives**

| # | Objective |
|---|-----------|
| 1 | Understand **why** ratio analysis removes scale effects and enables cross-sectional comparison. |
| 2 | Compute and interpret the five major ratio families: profitability, activity, liquidity, solvency, and valuation. |
| 3 | Decompose ROE into **3-factor** and **5-factor** DuPont trees. |
| 4 | Build radar charts, tornado charts, and multi-company dashboards entirely from scratch. |

---

## 1. Why Ratio Analysis?

### 1.1 Removing Scale Effects

Raw financial-statement numbers are difficult to compare across companies of different sizes.  A firm earning \$10 billion in revenue and \$1 billion in net income is not directly comparable with a firm earning \$500 million in revenue and \$75 million in net income.

**Ratio analysis** normalises financial data by expressing one line item as a fraction of another, producing **dimensionless** (or consistently dimensioned) metrics that allow:

- **Cross-sectional comparison** — benchmarking against peers within the same industry.
- **Time-series comparison** — tracking a single company's trajectory over multiple periods.
- **Standard-setting** — establishing minimum thresholds (e.g., loan covenants requiring a current ratio above 1.5).

### 1.2 The Five Families of Ratios

Financial ratios are conventionally grouped into five families:

| Family | Question Answered |
|--------|-------------------|
| **Profitability** | How effectively does the firm convert revenue into profit? |
| **Activity (efficiency)** | How effectively does the firm use its assets to generate revenue? |
| **Liquidity** | Can the firm meet its short-term obligations? |
| **Solvency** | Can the firm meet its long-term obligations? |
| **Valuation** | How does the market price the firm relative to fundamentals? |

> **Key Concept:** These five families span the entire balance sheet and income statement.  Together they provide a 360-degree view of a company's financial health.

### 1.3 Limitations of Ratio Analysis

> **Common Mistake:** Treating ratios as absolute truths without understanding context.  A current ratio of 1.2 might be healthy for a grocery chain (fast inventory turnover) but dangerously low for a capital-goods manufacturer.

Key limitations include:

1. **Accounting policy differences** — FIFO vs. LIFO, capitalisation vs. expensing, lease treatment.
2. **Industry heterogeneity** — ratios only make sense relative to the correct peer group.
3. **Seasonality** — balance-sheet snapshots on different dates may distort ratios.
4. **Historical cost** — asset values on the balance sheet may not reflect current market values.
5. **Earnings management** — management may use discretionary accruals to paint a rosier picture.
6. **One-time items** — restructuring charges, asset write-downs, and litigation settlements can distort single-period ratios.
7. **Conglomerate problem** — diversified firms operate in multiple industries, making peer selection difficult.

> **CFA Exam Tip:** The CFA curriculum emphasises that ratio analysis is a *starting point* for analysis, not an end in itself.  Always seek the economic story behind the numbers.

---

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Tolerances
ATOL = 1e-8
RTOL = 1e-6

# Colour palette
PRIMARY   = "steelblue"
SECONDARY = "coral"
TERTIARY  = "seagreen"
ACCENT    = "gold"

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Setup complete.")

### 2.1 Synthetic Financial-Statement Data

We create **three synthetic companies** — AlphaCorp, BetaInc, and GammaTech — each with five years (Year 1 through Year 5) of income-statement, balance-sheet, and market data stored as NumPy arrays.

All figures are in **millions of dollars**.

| Company | Profile | Revenue Range | Leverage |
|---------|---------|---------------|----------|
| **AlphaCorp** | Mature industrial | ~5,000-5,600 | Moderate |
| **BetaInc** | High-growth tech | ~2,000-5,200 | Low |
| **GammaTech** | Utility (stable) | ~8,000-8,300 | High |

> **Key Concept:** In practice you would pull these numbers from SEC filings (10-K / 10-Q) or a financial-data provider.  Here we generate plausible, internally consistent data so that every downstream ratio can be verified by hand.

In [ ]:
# ---------------------------------------------------------------
# Synthetic 3-statement data for 3 companies x 5 years
# All figures in $millions
# ---------------------------------------------------------------
years = np.arange(1, 6)  # Year 1 .. Year 5
n_years = len(years)

# === AlphaCorp (mature industrial, moderate leverage) ===
alpha = {}
alpha["name"] = "AlphaCorp"
alpha["revenue"]          = np.array([5000, 5200, 5500, 5350, 5600])
alpha["cogs"]             = np.array([3200, 3380, 3520, 3450, 3580])
alpha["sga"]              = np.array([ 600,  620,  650,  640,  660])
alpha["depreciation"]     = np.array([ 250,  260,  270,  265,  275])
alpha["interest_expense"] = np.array([  80,   78,   75,   77,   74])
alpha["tax_rate"]         = np.array([0.25, 0.25, 0.25, 0.25, 0.25])

# Derived income-statement items
alpha["gross_profit"]     = alpha["revenue"] - alpha["cogs"]
alpha["operating_income"] = alpha["gross_profit"] - alpha["sga"] - alpha["depreciation"]
alpha["ebt"]              = alpha["operating_income"] - alpha["interest_expense"]
alpha["tax"]              = alpha["ebt"] * alpha["tax_rate"]
alpha["net_income"]       = alpha["ebt"] - alpha["tax"]
alpha["ebitda"]           = alpha["operating_income"] + alpha["depreciation"]

# Balance-sheet items (end of period)
alpha["cash"]              = np.array([ 300,  320,  350,  330,  360])
alpha["receivables"]       = np.array([ 450,  470,  500,  480,  510])
alpha["inventory"]         = np.array([ 600,  620,  640,  610,  650])
alpha["other_current"]     = np.array([  50,   55,   60,   58,   62])
alpha["current_assets"]    = alpha["cash"] + alpha["receivables"] + alpha["inventory"] + alpha["other_current"]
alpha["ppe_net"]           = np.array([3000, 3100, 3200, 3150, 3250])
alpha["total_assets"]      = alpha["current_assets"] + alpha["ppe_net"]
alpha["current_liab"]      = np.array([ 800,  830,  860,  840,  870])
alpha["long_term_debt"]    = np.array([1200, 1150, 1100, 1120, 1080])
alpha["total_liab"]        = alpha["current_liab"] + alpha["long_term_debt"]
alpha["equity"]            = alpha["total_assets"] - alpha["total_liab"]

# Market data
alpha["shares_outstanding"] = np.array([100, 100, 100, 100, 100])  # millions
alpha["share_price"]        = np.array([ 32,  34,  37,  35,  39])
alpha["market_cap"]         = alpha["shares_outstanding"] * alpha["share_price"]

# Fixed charges (lease payments)
alpha["lease_payments"]     = np.array([40, 42, 44, 43, 45])

# Daily operating expenses (for defensive interval)
alpha["daily_op_exp"]       = (alpha["cogs"] + alpha["sga"]) / 365.0

# === BetaInc (high-growth tech, low leverage) ===
beta = {}
beta["name"] = "BetaInc"
beta["revenue"]          = np.array([2000, 2600, 3400, 4200, 5200])
beta["cogs"]             = np.array([ 600,  780, 1020, 1260, 1560])
beta["sga"]              = np.array([ 700,  850, 1050, 1200, 1400])
beta["depreciation"]     = np.array([ 100,  120,  150,  180,  210])
beta["interest_expense"] = np.array([  20,   22,   25,   28,   30])
beta["tax_rate"]         = np.array([0.22, 0.22, 0.22, 0.22, 0.22])

beta["gross_profit"]     = beta["revenue"] - beta["cogs"]
beta["operating_income"] = beta["gross_profit"] - beta["sga"] - beta["depreciation"]
beta["ebt"]              = beta["operating_income"] - beta["interest_expense"]
beta["tax"]              = beta["ebt"] * beta["tax_rate"]
beta["net_income"]       = beta["ebt"] - beta["tax"]
beta["ebitda"]           = beta["operating_income"] + beta["depreciation"]

beta["cash"]              = np.array([ 800,  900, 1100, 1300, 1600])
beta["receivables"]       = np.array([ 200,  260,  340,  420,  520])
beta["inventory"]         = np.array([  80,  100,  130,  160,  200])
beta["other_current"]     = np.array([  20,   25,   30,   35,   40])
beta["current_assets"]    = beta["cash"] + beta["receivables"] + beta["inventory"] + beta["other_current"]
beta["ppe_net"]           = np.array([ 800, 1000, 1300, 1600, 2000])
beta["total_assets"]      = beta["current_assets"] + beta["ppe_net"]
beta["current_liab"]      = np.array([ 300,  370,  460,  550,  660])
beta["long_term_debt"]    = np.array([ 200,  220,  250,  280,  300])
beta["total_liab"]        = beta["current_liab"] + beta["long_term_debt"]
beta["equity"]            = beta["total_assets"] - beta["total_liab"]

beta["shares_outstanding"] = np.array([200, 200, 200, 200, 200])
beta["share_price"]        = np.array([ 18,  24,  33,  42,  55])
beta["market_cap"]         = beta["shares_outstanding"] * beta["share_price"]

beta["lease_payments"]     = np.array([15, 18, 22, 26, 30])
beta["daily_op_exp"]       = (beta["cogs"] + beta["sga"]) / 365.0

# === GammaTech (utility, high leverage, stable) ===
gamma = {}
gamma["name"] = "GammaTech"
gamma["revenue"]          = np.array([8000, 8100, 8200, 8150, 8300])
gamma["cogs"]             = np.array([5200, 5300, 5350, 5320, 5400])
gamma["sga"]              = np.array([ 800,  810,  820,  815,  830])
gamma["depreciation"]     = np.array([ 500,  510,  520,  515,  525])
gamma["interest_expense"] = np.array([ 300,  295,  290,  292,  285])
gamma["tax_rate"]         = np.array([0.28, 0.28, 0.28, 0.28, 0.28])

gamma["gross_profit"]     = gamma["revenue"] - gamma["cogs"]
gamma["operating_income"] = gamma["gross_profit"] - gamma["sga"] - gamma["depreciation"]
gamma["ebt"]              = gamma["operating_income"] - gamma["interest_expense"]
gamma["tax"]              = gamma["ebt"] * gamma["tax_rate"]
gamma["net_income"]       = gamma["ebt"] - gamma["tax"]
gamma["ebitda"]           = gamma["operating_income"] + gamma["depreciation"]

gamma["cash"]              = np.array([ 400,  410,  420,  415,  430])
gamma["receivables"]       = np.array([ 700,  710,  720,  715,  730])
gamma["inventory"]         = np.array([ 900,  910,  920,  915,  930])
gamma["other_current"]     = np.array([  80,   82,   85,   83,   88])
gamma["current_assets"]    = gamma["cash"] + gamma["receivables"] + gamma["inventory"] + gamma["other_current"]
gamma["ppe_net"]           = np.array([6000, 6050, 6100, 6080, 6150])
gamma["total_assets"]      = gamma["current_assets"] + gamma["ppe_net"]
gamma["current_liab"]      = np.array([1200, 1210, 1220, 1215, 1230])
gamma["long_term_debt"]    = np.array([3500, 3450, 3400, 3420, 3380])
gamma["total_liab"]        = gamma["current_liab"] + gamma["long_term_debt"]
gamma["equity"]            = gamma["total_assets"] - gamma["total_liab"]

gamma["shares_outstanding"] = np.array([500, 500, 500, 500, 500])
gamma["share_price"]        = np.array([ 15,  15,  16,  15,  16])
gamma["market_cap"]         = gamma["shares_outstanding"] * gamma["share_price"]

gamma["lease_payments"]     = np.array([60, 62, 64, 63, 65])
gamma["daily_op_exp"]       = (gamma["cogs"] + gamma["sga"]) / 365.0

companies = [alpha, beta, gamma]
company_names = [c["name"] for c in companies]

print("Synthetic data created for:", ", ".join(company_names))
print(f"Each company has {n_years} years of data (Year 1 to Year 5).")

## 3. Profitability Ratios

Profitability ratios measure a firm's ability to generate profit relative to revenue, assets, or equity.  They answer the fundamental question: *"How effectively does the firm convert inputs into bottom-line returns?"*

### 3.1 Gross Profit Margin

$$
\text{Gross Margin} = \frac{\text{Revenue} - \text{COGS}}{\text{Revenue}} = \frac{\text{Gross Profit}}{\text{Revenue}}
$$

**Interpretation:** The percentage of each revenue dollar remaining after paying for the direct cost of goods sold.  A high gross margin indicates pricing power or low production costs.

> **Key Concept:** Gross margin is the *first* profitability gate.  If gross margin is thin, there is little room for operating expenses, interest, and taxes to be covered downstream.

### 3.2 Operating Profit Margin

$$
\text{Operating Margin} = \frac{\text{Operating Income (EBIT)}}{\text{Revenue}}
$$

**Interpretation:** Captures both production efficiency **and** control of operating expenses (SG&A, depreciation).  It strips out financing decisions and tax effects, making it ideal for comparing firms with different capital structures.

### 3.3 Net Profit Margin

$$
\text{Net Margin} = \frac{\text{Net Income}}{\text{Revenue}}
$$

**Interpretation:** The *bottom-line* percentage.  Includes all costs — operating, financing, and tax.  Useful as a comprehensive profitability gauge, but sensitive to capital-structure and tax-regime differences.

### 3.4 Return on Assets (ROA)

$$
\text{ROA} = \frac{\text{Net Income}}{\text{Total Assets}}
$$

**Interpretation:** How much profit the firm generates per dollar of assets, regardless of how those assets are financed.

> **CFA Exam Tip:** Some textbooks define ROA using *average* total assets to smooth out intra-year balance-sheet fluctuations.  The CFA curriculum accepts both end-of-period and average formulations — just be consistent.

### 3.5 Return on Equity (ROE)

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Shareholders' Equity}}
$$

**Interpretation:** The return earned on owners' capital.  ROE is the single most important ratio for equity investors and is the ratio that the DuPont decomposition dissects.

> **Common Mistake:** A very high ROE can be driven by excessive leverage rather than genuine operating efficiency.  Always check whether high ROE comes from high margins or high leverage — the DuPont framework (Section 8) addresses exactly this.

In [ ]:
# ---------------------------------------------------------------
# 3. Profitability Ratios — compute for all companies
# ---------------------------------------------------------------

def profitability_ratios(c):
    """Return dict of profitability ratios for a company dict."""
    return {
        "Gross Margin":     c["gross_profit"] / c["revenue"],
        "Operating Margin": c["operating_income"] / c["revenue"],
        "Net Margin":       c["net_income"] / c["revenue"],
        "ROA":              c["net_income"] / c["total_assets"],
        "ROE":              c["net_income"] / c["equity"],
    }

# Compute and display
for c in companies:
    ratios = profitability_ratios(c)
    print(f"\n{'='*55}")
    print(f"  {c['name']} — Profitability Ratios")
    print(f"{'='*55}")
    print(f"{'Ratio':<22}", end="")
    for y in years:
        print(f"  Yr{y:d}", end="")
    print()
    print("-" * 55)
    for name, vals in ratios.items():
        print(f"{name:<22}", end="")
        for v in vals:
            print(f" {v:6.1%}", end="")
        print()

In [ ]:
# ---------------------------------------------------------------
# Visualise: Net Margin trend for all 3 companies
# ---------------------------------------------------------------
fig, ax = plt.subplots()
colours = [PRIMARY, SECONDARY, TERTIARY]
for c, col in zip(companies, colours):
    margin = c["net_income"] / c["revenue"]
    ax.plot(years, margin * 100, marker="o", color=col, linewidth=2, label=c["name"])

ax.set_xlabel("Year")
ax.set_ylabel("Net Profit Margin (%)")
ax.set_title("Net Profit Margin — 5-Year Trend")
ax.set_xticks(years)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Activity (Efficiency) Ratios

Activity ratios gauge how efficiently a firm deploys its assets to generate revenue.  They bridge the income statement and the balance sheet.

### 4.1 Receivables Turnover & Days Sales Outstanding

$$
\text{Receivables Turnover} = \frac{\text{Revenue}}{\text{Accounts Receivable}}
$$

$$
\text{DSO} = \frac{365}{\text{Receivables Turnover}}
$$

**Interpretation:** How many times per year the firm collects its average receivables balance.  A *higher* turnover (lower DSO) means faster collection.

> **Key Concept:** Unusually low receivables turnover may signal lax credit policies or difficulty collecting from customers — both red flags for cash-flow quality.

### 4.2 Inventory Turnover & Days Inventory on Hand

$$
\text{Inventory Turnover} = \frac{\text{COGS}}{\text{Inventory}}
$$

$$
\text{DIH} = \frac{365}{\text{Inventory Turnover}}
$$

**Interpretation:** The number of times inventory is sold and replaced during the year.  Higher turnover means less capital is tied up in inventory.

### 4.3 Total Asset Turnover

$$
\text{Total Asset Turnover} = \frac{\text{Revenue}}{\text{Total Assets}}
$$

**Interpretation:** Revenue generated per dollar of assets.  This ratio is a **key driver** in the DuPont decomposition.

### 4.4 Fixed Asset Turnover

$$
\text{Fixed Asset Turnover} = \frac{\text{Revenue}}{\text{Net PP\&E}}
$$

**Interpretation:** How efficiently the firm uses its long-lived tangible assets to generate sales.  Capital-intensive industries (utilities, manufacturing) tend to have lower fixed-asset turnover.

> **CFA Exam Tip:** When comparing firms, ensure consistency in accounting for leases — capitalised leases inflate PP&E and total assets, reducing turnover ratios.

In [ ]:
# ---------------------------------------------------------------
# 4. Activity (Efficiency) Ratios
# ---------------------------------------------------------------

def activity_ratios(c):
    recv_turnover = c["revenue"] / c["receivables"]
    inv_turnover  = c["cogs"] / c["inventory"]
    return {
        "Receivables Turnover": recv_turnover,
        "DSO (days)":           365.0 / recv_turnover,
        "Inventory Turnover":   inv_turnover,
        "DIH (days)":           365.0 / inv_turnover,
        "Total Asset Turnover": c["revenue"] / c["total_assets"],
        "Fixed Asset Turnover": c["revenue"] / c["ppe_net"],
    }

for c in companies:
    ratios = activity_ratios(c)
    print(f"\n{'='*60}")
    print(f"  {c['name']} — Activity Ratios")
    print(f"{'='*60}")
    print(f"{'Ratio':<24}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 60)
    for name, vals in ratios.items():
        print(f"{name:<24}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

In [ ]:
# ---------------------------------------------------------------
# Visualise: Total Asset Turnover comparison
# ---------------------------------------------------------------
fig, ax = plt.subplots()
width = 0.25
x = np.arange(n_years)

for i, (c, col) in enumerate(zip(companies, colours)):
    tat = c["revenue"] / c["total_assets"]
    ax.bar(x + i * width, tat, width, color=col, label=c["name"], edgecolor="white")

ax.set_xlabel("Year")
ax.set_ylabel("Total Asset Turnover (x)")
ax.set_title("Total Asset Turnover — Cross-Sectional Comparison")
ax.set_xticks(x + width)
ax.set_xticklabels([f"Yr {y}" for y in years])
ax.legend()
plt.tight_layout()
plt.show()

## 5. Liquidity Ratios

Liquidity ratios assess whether a firm can meet its **short-term obligations** as they come due.

### 5.1 Current Ratio

$$
\text{Current Ratio} = \frac{\text{Current Assets}}{\text{Current Liabilities}}
$$

**Interpretation:** The broadest liquidity measure.  A ratio above 1.0 implies current assets exceed current liabilities.

> **Common Mistake:** A very *high* current ratio is not always good — it may indicate excessive inventory or idle cash that is not being deployed productively.

### 5.2 Quick Ratio (Acid-Test)

$$
\text{Quick Ratio} = \frac{\text{Cash} + \text{Receivables}}{\text{Current Liabilities}}
$$

**Interpretation:** A stricter test that removes inventory (the least liquid current asset) from the numerator.

### 5.3 Cash Ratio

$$
\text{Cash Ratio} = \frac{\text{Cash}}{\text{Current Liabilities}}
$$

**Interpretation:** The most conservative liquidity measure — only cash is counted.

### 5.4 Defensive Interval Ratio

$$
\text{DIR} = \frac{\text{Cash} + \text{Receivables} + \text{Marketable Securities}}{\text{Daily Operating Expenditures}}
$$

**Interpretation:** The number of days a firm can operate using only its liquid assets, without any additional revenue.  A longer interval provides a greater cushion.

> **Key Concept:** Liquidity ratios are *static* snapshots.  They do not capture the *timing* of cash inflows and outflows within the period.  Cash-flow analysis complements ratio analysis for a complete liquidity picture.

In [ ]:
# ---------------------------------------------------------------
# 5. Liquidity Ratios
# ---------------------------------------------------------------

def liquidity_ratios(c):
    return {
        "Current Ratio":   c["current_assets"] / c["current_liab"],
        "Quick Ratio":     (c["cash"] + c["receivables"]) / c["current_liab"],
        "Cash Ratio":      c["cash"] / c["current_liab"],
        "Defensive Interval (days)": (c["cash"] + c["receivables"]) / c["daily_op_exp"],
    }

for c in companies:
    ratios = liquidity_ratios(c)
    print(f"\n{'='*62}")
    print(f"  {c['name']} — Liquidity Ratios")
    print(f"{'='*62}")
    print(f"{'Ratio':<30}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 62)
    for name, vals in ratios.items():
        print(f"{name:<30}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

In [ ]:
# ---------------------------------------------------------------
# Visualise: Liquidity ratios for latest year
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

labels = company_names
current = [c["current_assets"][-1] / c["current_liab"][-1] for c in companies]
quick   = [(c["cash"][-1] + c["receivables"][-1]) / c["current_liab"][-1] for c in companies]
cash_r  = [c["cash"][-1] / c["current_liab"][-1] for c in companies]

x = np.arange(len(labels))
w = 0.22
ax.bar(x - w, current, w, color=PRIMARY, label="Current Ratio")
ax.bar(x,     quick,   w, color=SECONDARY, label="Quick Ratio")
ax.bar(x + w, cash_r,  w, color=TERTIARY, label="Cash Ratio")

ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="1.0 threshold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Ratio (x)")
ax.set_title("Liquidity Ratios — Year 5 Snapshot")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Solvency Ratios

Solvency ratios evaluate a firm's capacity to service its **long-term** debt obligations.

### 6.1 Debt-to-Equity Ratio

$$
\text{D/E} = \frac{\text{Total Liabilities}}{\text{Shareholders' Equity}}
$$

**Interpretation:** The proportion of creditor financing relative to owner financing.  Higher D/E indicates greater financial leverage and, consequently, greater financial risk.

### 6.2 Debt-to-Assets Ratio

$$
\text{D/A} = \frac{\text{Total Liabilities}}{\text{Total Assets}}
$$

**Interpretation:** The fraction of assets financed by debt.  A D/A above 0.5 means creditors have provided more financing than equity holders.

> **Key Concept:** Leverage amplifies both returns and risk.  During expansions, leverage boosts ROE; during contractions, it magnifies losses.  This trade-off is at the heart of capital-structure theory (Modigliani-Miller).

### 6.3 Interest Coverage Ratio

$$
\text{Interest Coverage} = \frac{\text{EBIT}}{\text{Interest Expense}}
$$

**Interpretation:** The number of times operating income covers interest payments.  A ratio below 1.5 is a strong warning signal; bond covenants often require coverage above 2.0 or 3.0.

### 6.4 Fixed Charge Coverage Ratio

$$
\text{Fixed Charge Coverage} = \frac{\text{EBIT} + \text{Lease Payments}}{\text{Interest Expense} + \text{Lease Payments}}
$$

**Interpretation:** Broadens interest coverage to include all fixed financial commitments (e.g., lease obligations).

> **CFA Exam Tip:** The CFA curriculum tests the fixed-charge coverage ratio specifically — do not skip it.

In [ ]:
# ---------------------------------------------------------------
# 6. Solvency Ratios
# ---------------------------------------------------------------

def solvency_ratios(c):
    return {
        "Debt / Equity":        c["total_liab"] / c["equity"],
        "Debt / Assets":        c["total_liab"] / c["total_assets"],
        "Interest Coverage":    c["operating_income"] / c["interest_expense"],
        "Fixed Charge Coverage": (c["operating_income"] + c["lease_payments"]) /
                                 (c["interest_expense"] + c["lease_payments"]),
    }

for c in companies:
    ratios = solvency_ratios(c)
    print(f"\n{'='*62}")
    print(f"  {c['name']} — Solvency Ratios")
    print(f"{'='*62}")
    print(f"{'Ratio':<26}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 62)
    for name, vals in ratios.items():
        print(f"{name:<26}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

In [ ]:
# ---------------------------------------------------------------
# Visualise: Debt-to-Equity trend
# ---------------------------------------------------------------
fig, ax = plt.subplots()
for c, col in zip(companies, colours):
    de = c["total_liab"] / c["equity"]
    ax.plot(years, de, marker="s", color=col, linewidth=2, label=c["name"])

ax.set_xlabel("Year")
ax.set_ylabel("Debt / Equity")
ax.set_title("Leverage (D/E) — 5-Year Trend")
ax.set_xticks(years)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Valuation Ratios (Preview)

Valuation ratios relate a firm's **market price** to its fundamentals.  They are covered in depth under Equity Valuation; here we present a brief overview as a bridge from financial-statement analysis to security analysis.

### 7.1 Price-to-Earnings (P/E)

$$
\text{P/E} = \frac{\text{Market Price per Share}}{\text{Earnings per Share}}
$$

**Interpretation:** How much investors are willing to pay per dollar of current earnings.  High P/E can indicate growth expectations or overvaluation.

### 7.2 Price-to-Book (P/B)

$$
\text{P/B} = \frac{\text{Market Price per Share}}{\text{Book Value per Share}}
$$

**Interpretation:** Compares market valuation to accounting (book) value.  A P/B below 1.0 may signal undervaluation or fundamental weakness.

### 7.3 Price-to-Sales (P/S)

$$
\text{P/S} = \frac{\text{Market Cap}}{\text{Revenue}}
$$

**Interpretation:** Useful for firms with negative earnings where P/E is undefined.

### 7.4 EV/EBITDA

$$
\text{EV/EBITDA} = \frac{\text{Enterprise Value}}{\text{EBITDA}}
$$

where Enterprise Value = Market Cap + Total Debt - Cash.

> **Key Concept:** EV/EBITDA is capital-structure neutral (unlike P/E), making it a preferred multiple for comparing companies with different leverage levels.

In [ ]:
# ---------------------------------------------------------------
# 7. Valuation Ratios
# ---------------------------------------------------------------

def valuation_ratios(c):
    eps = c["net_income"] / c["shares_outstanding"]
    bvps = c["equity"] / c["shares_outstanding"]
    ev = c["market_cap"] + c["long_term_debt"] - c["cash"]
    return {
        "EPS ($)":       eps,
        "P/E":           c["share_price"] / eps,
        "P/B":           c["share_price"] / bvps,
        "P/S":           c["market_cap"] / c["revenue"],
        "EV/EBITDA":     ev / c["ebitda"],
    }

for c in companies:
    ratios = valuation_ratios(c)
    print(f"\n{'='*55}")
    print(f"  {c['name']} — Valuation Ratios")
    print(f"{'='*55}")
    print(f"{'Ratio':<16}", end="")
    for y in years:
        print(f"   Yr{y:d}", end="")
    print()
    print("-" * 55)
    for name, vals in ratios.items():
        print(f"{name:<16}", end="")
        for v in vals:
            print(f" {v:7.2f}", end="")
        print()

## 8. DuPont 3-Factor Decomposition

### 8.1 The Core Insight

Return on Equity is the product of three distinct drivers:

$$
\text{ROE} = \underbrace{\frac{\text{Net Income}}{\text{Revenue}}}_{\text{Net Profit Margin}} \;\times\; \underbrace{\frac{\text{Revenue}}{\text{Total Assets}}}_{\text{Asset Turnover}} \;\times\; \underbrace{\frac{\text{Total Assets}}{\text{Equity}}}_{\text{Equity Multiplier (Leverage)}}
$$

### 8.2 Algebraic Derivation

Start from the definition of ROE:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Equity}}
$$

Multiply and divide by Revenue and Total Assets:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Equity}} \times \frac{\text{Revenue}}{\text{Revenue}} \times \frac{\text{Total Assets}}{\text{Total Assets}}
$$

Rearranging:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} \times \frac{\text{Total Assets}}{\text{Equity}}
$$

This is precisely the 3-factor DuPont identity.

> **Key Concept:** The DuPont decomposition reveals *where* ROE comes from.  Two companies can have identical ROEs but very different profiles — one may rely on high margins, the other on heavy leverage.

### 8.3 Interpreting the Three Factors

| Factor | Measures | Improved by |
|--------|----------|-------------|
| **Net Profit Margin** | Profitability | Raising prices, cutting costs, tax efficiency |
| **Asset Turnover** | Efficiency | Generating more revenue per dollar of assets |
| **Equity Multiplier** | Leverage | Using more debt (increases risk) |

> **CFA Exam Tip:** A rising ROE driven primarily by an increasing equity multiplier signals growing financial risk, not improving operations.

The beauty of the DuPont framework is that it converts a single opaque number (ROE) into an *actionable diagnostic*.  Management teams and analysts can identify which lever to pull:

- **Low margin?** Focus on cost reduction or pricing strategy.
- **Low turnover?** Improve asset utilisation — reduce excess inventory, collect receivables faster.
- **High multiplier?** The ROE may be artificially inflated by leverage; check solvency ratios.

In [ ]:
# ---------------------------------------------------------------
# 8. DuPont 3-Factor Decomposition
# ---------------------------------------------------------------

def dupont_3(c):
    npm = c["net_income"] / c["revenue"]            # Net profit margin
    ato = c["revenue"] / c["total_assets"]           # Asset turnover
    em  = c["total_assets"] / c["equity"]            # Equity multiplier
    roe = npm * ato * em                             # Should equal NI / Equity
    return npm, ato, em, roe

print(f"{'Company':<12} {'Year':>4}  {'NPM':>7} {'ATO':>7} {'EM':>7}  {'ROE (DuPont)':>13} {'ROE (Direct)':>13}  {'Match':>5}")
print("-" * 80)

for c in companies:
    npm, ato, em, roe_dp = dupont_3(c)
    roe_direct = c["net_income"] / c["equity"]
    for i, y in enumerate(years):
        match = np.isclose(roe_dp[i], roe_direct[i], atol=ATOL)
        print(f"{c['name']:<12} {y:4d}  {npm[i]:7.2%} {ato[i]:7.3f} {em[i]:7.3f}  {roe_dp[i]:13.4%} {roe_direct[i]:13.4%}  {'OK' if match else 'FAIL':>5}")

In [ ]:
# ---------------------------------------------------------------
# Visualise: DuPont tree for AlphaCorp Year 5
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("DuPont 3-Factor Tree — AlphaCorp, Year 5", fontsize=14, fontweight="bold")

npm_a, ato_a, em_a, roe_a = dupont_3(alpha)

# Positions: (x, y)
boxes = {
    "ROE":    (5, 8.5, f"ROE\n{roe_a[-1]:.2%}"),
    "NPM":    (2, 6, f"Net Profit Margin\n{npm_a[-1]:.2%}"),
    "ATO":    (5, 6, f"Asset Turnover\n{ato_a[-1]:.3f}x"),
    "EM":     (8, 6, f"Equity Multiplier\n{em_a[-1]:.3f}x"),
    "NI":     (1, 3.5, f"Net Income\n${alpha['net_income'][-1]:,.0f}M"),
    "Rev1":   (3, 3.5, f"Revenue\n${alpha['revenue'][-1]:,.0f}M"),
    "Rev2":   (4, 3.5, f"Revenue\n${alpha['revenue'][-1]:,.0f}M"),
    "TA1":    (6, 3.5, f"Total Assets\n${alpha['total_assets'][-1]:,.0f}M"),
    "TA2":    (7, 3.5, f"Total Assets\n${alpha['total_assets'][-1]:,.0f}M"),
    "EQ":     (9, 3.5, f"Equity\n${alpha['equity'][-1]:,.0f}M"),
}

box_style = dict(boxstyle="round,pad=0.4", facecolor="lightyellow", edgecolor="grey")
for key, (bx, by, txt) in boxes.items():
    ax.text(bx, by, txt, ha="center", va="center", fontsize=9, bbox=box_style)

# Connectors
connectors = [
    ("ROE", "NPM"), ("ROE", "ATO"), ("ROE", "EM"),
    ("NPM", "NI"), ("NPM", "Rev1"),
    ("ATO", "Rev2"), ("ATO", "TA1"),
    ("EM", "TA2"), ("EM", "EQ"),
]
for parent, child in connectors:
    px, py, _ = boxes[parent]
    cx, cy, _ = boxes[child]
    ax.annotate("", xy=(cx, cy + 0.6), xytext=(px, py - 0.6),
                arrowprops=dict(arrowstyle="->", color="grey", lw=1.2))

# Multiplication signs
ax.text(3.5, 6, r"$\times$", ha="center", va="center", fontsize=16, color="red")
ax.text(6.5, 6, r"$\times$", ha="center", va="center", fontsize=16, color="red")

plt.tight_layout()
plt.show()

## 9. DuPont 5-Factor Decomposition

### 9.1 Extending the Framework

The 3-factor model lumps together tax effects, interest burden, and operating profitability into a single "net profit margin" term.  The **5-factor DuPont** model unpacks net profit margin into three sub-components:

$$
\text{ROE} = \underbrace{\frac{\text{Net Income}}{\text{EBT}}}_{\text{Tax Burden}} \;\times\; \underbrace{\frac{\text{EBT}}{\text{EBIT}}}_{\text{Interest Burden}} \;\times\; \underbrace{\frac{\text{EBIT}}{\text{Revenue}}}_{\text{Operating Margin}} \;\times\; \underbrace{\frac{\text{Revenue}}{\text{Total Assets}}}_{\text{Asset Turnover}} \;\times\; \underbrace{\frac{\text{Total Assets}}{\text{Equity}}}_{\text{Equity Multiplier}}
$$

### 9.2 Algebraic Derivation

Begin with the 3-factor identity:

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Revenue}} \times \frac{\text{Revenue}}{\text{Total Assets}} \times \frac{\text{Total Assets}}{\text{Equity}}
$$

Now decompose Net Profit Margin by multiplying and dividing by EBT and EBIT:

$$
\frac{\text{Net Income}}{\text{Revenue}} = \frac{\text{Net Income}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Revenue}}
$$

Substituting back yields the full 5-factor decomposition:

$$
\text{ROE} = \frac{\text{NI}}{\text{EBT}} \times \frac{\text{EBT}}{\text{EBIT}} \times \frac{\text{EBIT}}{\text{Rev}} \times \frac{\text{Rev}}{\text{Assets}} \times \frac{\text{Assets}}{\text{Equity}}
$$

### 9.3 Interpreting the Five Factors

| Factor | Formula | Measures | Range |
|--------|---------|----------|-------|
| **Tax Burden** | NI / EBT | How much EBT survives taxation | 0 to 1 (higher = lower tax rate) |
| **Interest Burden** | EBT / EBIT | How much EBIT survives interest costs | 0 to 1 (higher = lower interest cost) |
| **Operating Margin** | EBIT / Revenue | Core operational profitability | varies by industry |
| **Asset Turnover** | Revenue / Assets | Capital efficiency | varies by industry |
| **Equity Multiplier** | Assets / Equity | Financial leverage | >= 1 (higher = more debt) |

> **Key Concept:** The 5-factor model lets you pinpoint *exactly* why ROE changed.  Did the tax rate go up?  Did interest expense grow?  Did operating efficiency improve?  Each question maps to one factor.

> **Common Mistake:** Confusing the "interest burden" direction — a *lower* interest burden ratio means *more* of EBIT is consumed by interest, which is *worse*, not better.

### 9.4 Practical Use Cases

The 5-factor DuPont is especially useful for:

- **Year-over-year analysis:** If ROE dropped, which factor drove the decline?
- **Cross-border comparison:** Companies in different tax jurisdictions will differ in the tax-burden factor.
- **Leverage analysis:** Separating the interest burden from the equity multiplier reveals the *cost* of leverage, not just its presence.

> **CFA Exam Tip:** The CFA Level 1 curriculum explicitly tests the 5-factor decomposition.  Practice computing all five factors from a given set of financial statements.

In [ ]:
# ---------------------------------------------------------------
# 9. DuPont 5-Factor Decomposition
# ---------------------------------------------------------------

def dupont_5(c):
    tax_burden      = c["net_income"] / c["ebt"]                # NI / EBT
    interest_burden = c["ebt"] / c["operating_income"]           # EBT / EBIT
    operating_margin = c["operating_income"] / c["revenue"]      # EBIT / Revenue
    asset_turnover  = c["revenue"] / c["total_assets"]           # Revenue / Assets
    equity_mult     = c["total_assets"] / c["equity"]            # Assets / Equity
    roe = tax_burden * interest_burden * operating_margin * asset_turnover * equity_mult
    return {
        "Tax Burden":       tax_burden,
        "Interest Burden":  interest_burden,
        "Operating Margin": operating_margin,
        "Asset Turnover":   asset_turnover,
        "Equity Multiplier": equity_mult,
        "ROE (5-factor)":   roe,
    }

for c in companies:
    factors = dupont_5(c)
    roe_direct = c["net_income"] / c["equity"]
    print(f"\n{'='*70}")
    print(f"  {c['name']} — 5-Factor DuPont Decomposition")
    print(f"{'='*70}")
    print(f"{'Factor':<22}", end="")
    for y in years:
        print(f"    Yr{y:d}", end="")
    print()
    print("-" * 70)
    for name, vals in factors.items():
        print(f"{name:<22}", end="")
        for v in vals:
            print(f" {v:7.4f}", end="")
        print()
    # Verification
    print(f"{'ROE (direct)':<22}", end="")
    for v in roe_direct:
        print(f" {v:7.4f}", end="")
    print()
    match_all = np.allclose(factors["ROE (5-factor)"], roe_direct, atol=ATOL)
    print(f"  Verification: {'PASS' if match_all else 'FAIL'}")

In [ ]:
# ---------------------------------------------------------------
# Tornado chart: sensitivity of ROE to each DuPont factor (Year 5)
# ---------------------------------------------------------------

def tornado_dupont(c, year_idx=-1):
    """
    For each DuPont factor, compute how much ROE would change
    if that factor increased by 10%, holding all others constant.
    """
    factors_dict = dupont_5(c)
    factor_names = ["Tax Burden", "Interest Burden", "Operating Margin",
                    "Asset Turnover", "Equity Multiplier"]
    base_roe = factors_dict["ROE (5-factor)"][year_idx]
    base_vals = [factors_dict[f][year_idx] for f in factor_names]

    deltas = []
    for i in range(5):
        shocked = base_vals.copy()
        shocked[i] *= 1.10  # +10%
        new_roe = np.prod(shocked)
        deltas.append(new_roe - base_roe)

    return factor_names, deltas, base_roe


fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, c, col in zip(axes, companies, colours):
    names, deltas, base = tornado_dupont(c)
    # Sort by absolute impact
    order = np.argsort(np.abs(deltas))
    sorted_names = [names[i] for i in order]
    sorted_deltas = [deltas[i] for i in order]

    bars = ax.barh(sorted_names, [d * 100 for d in sorted_deltas], color=col, edgecolor="white")
    ax.set_xlabel("Change in ROE (pp)")
    ax.set_title(f"{c['name']}\nBase ROE = {base:.2%}")
    ax.axvline(0, color="black", linewidth=0.8)

fig.suptitle("Tornado Chart — ROE Sensitivity to +10% Shock in Each DuPont Factor (Year 5)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 10. Cross-Sectional vs. Time-Series Analysis

### 10.1 Two Dimensions of Comparison

Ratio analysis is most powerful when applied along **two axes**:

1. **Cross-sectional (peer comparison):** Comparing the same ratio across multiple companies *at a single point in time*.  This answers: "Is this company better or worse than its peers?"
2. **Time-series (trend analysis):** Tracking one company's ratio over multiple periods.  This answers: "Is this company improving or deteriorating?"

### 10.2 Methodology

- **Select comparable companies** — same industry, similar size, same accounting standards.
- **Normalise for accounting differences** — adjust for FIFO/LIFO, lease capitalisation, one-time items.
- **Use median (not mean) for peer benchmarks** — medians are robust to outliers.

### 10.3 Pitfalls

> **Common Mistake:** Comparing a software company's asset turnover with that of a utility company.  Industry composition dominates most ratios, so cross-sectional analysis must be done *within* a well-defined peer group.

Additional pitfalls:

- **Survivorship bias** — if you only look at companies that survived to Year 5, the sample is upward-biased.
- **Changing accounting standards** — IFRS 16 (lease capitalisation) significantly altered leverage and asset ratios.
- **Mergers and divestitures** — can cause discontinuities in time-series data.

> **CFA Exam Tip:** The curriculum specifically asks candidates to distinguish between cross-sectional and time-series analysis and describe the limitations of each.

### 10.4 Radar (Spider) Charts

A radar chart is a popular way to visualise multiple ratios simultaneously for several companies.  Each spoke represents a ratio, and each company is a polygon.  The further a vertex is from the centre, the "better" (or higher) that ratio is.

> **Key Concept:** Radar charts are useful for *qualitative* comparison — they give an at-a-glance profile — but can be misleading if the scales are not normalised.  We normalise each ratio to the [0, 1] range across the companies being compared.

In [ ]:
# ---------------------------------------------------------------
# 10. Radar (Spider) Chart — Cross-Sectional Comparison (Year 5)
# ---------------------------------------------------------------

def compute_summary_ratios(c, idx=-1):
    """Return a dict of key ratios for a single year."""
    return {
        "Net Margin":        c["net_income"][idx] / c["revenue"][idx],
        "ROE":               c["net_income"][idx] / c["equity"][idx],
        "Asset Turnover":    c["revenue"][idx] / c["total_assets"][idx],
        "Current Ratio":     c["current_assets"][idx] / c["current_liab"][idx],
        "Interest Coverage": c["operating_income"][idx] / c["interest_expense"][idx],
        "D/E":               c["total_liab"][idx] / c["equity"][idx],
    }

# Gather data
ratio_names = list(compute_summary_ratios(companies[0]).keys())
n_ratios = len(ratio_names)

raw_values = []
for c in companies:
    vals = compute_summary_ratios(c)
    raw_values.append([vals[r] for r in ratio_names])

raw_values = np.array(raw_values)  # (3, n_ratios)

# Normalise to [0, 1] across companies for each ratio
# For D/E, lower is "better", so invert
invert = [False, False, False, False, False, True]  # D/E inverted
normed = np.zeros_like(raw_values)
for j in range(n_ratios):
    col = raw_values[:, j]
    if invert[j]:
        col = -col  # invert so higher = better
    mn, mx = col.min(), col.max()
    if mx - mn > 1e-12:
        normed[:, j] = (col - mn) / (mx - mn)
    else:
        normed[:, j] = 0.5

# Radar chart
angles = np.linspace(0, 2 * np.pi, n_ratios, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (c, col) in enumerate(zip(companies, colours)):
    values = normed[i].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", color=col, linewidth=2, label=c["name"])
    ax.fill(angles, values, color=col, alpha=0.1)

ax.set_thetagrids(np.degrees(angles[:-1]), ratio_names, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title("Radar Chart — Cross-Sectional Comparison (Year 5)\n(normalised to [0,1]; D/E inverted so outward = better)",
             fontsize=12, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## 11. Multi-Company Dashboard

### 11.1 Comprehensive Ratio Table

Below we compile all major ratios into a single table for easy comparison across companies and years.

> **Key Concept:** Dashboards should prioritise *clarity* over *density*.  A well-designed dashboard tells a story; a cluttered one obscures it.

In [ ]:
# ---------------------------------------------------------------
# 11.1 Comprehensive Ratio Comparison Table (Year 5)
# ---------------------------------------------------------------

def all_ratios_year(c, idx=-1):
    """Return an ordered dict of all ratios for a single year."""
    rev = c["revenue"][idx]
    ni  = c["net_income"][idx]
    ta  = c["total_assets"][idx]
    eq  = c["equity"][idx]
    eps = ni / c["shares_outstanding"][idx]
    bvps = eq / c["shares_outstanding"][idx]
    ev = c["market_cap"][idx] + c["long_term_debt"][idx] - c["cash"][idx]

    ratios = {}
    # Profitability
    ratios["Gross Margin"]      = c["gross_profit"][idx] / rev
    ratios["Operating Margin"]  = c["operating_income"][idx] / rev
    ratios["Net Margin"]        = ni / rev
    ratios["ROA"]               = ni / ta
    ratios["ROE"]               = ni / eq
    # Activity
    ratios["Receivables TO"]    = rev / c["receivables"][idx]
    ratios["Inventory TO"]      = c["cogs"][idx] / c["inventory"][idx]
    ratios["Total Asset TO"]    = rev / ta
    ratios["Fixed Asset TO"]    = rev / c["ppe_net"][idx]
    # Liquidity
    ratios["Current Ratio"]     = c["current_assets"][idx] / c["current_liab"][idx]
    ratios["Quick Ratio"]       = (c["cash"][idx] + c["receivables"][idx]) / c["current_liab"][idx]
    ratios["Cash Ratio"]        = c["cash"][idx] / c["current_liab"][idx]
    # Solvency
    ratios["D/E"]               = c["total_liab"][idx] / eq
    ratios["D/A"]               = c["total_liab"][idx] / ta
    ratios["Interest Coverage"] = c["operating_income"][idx] / c["interest_expense"][idx]
    # Valuation
    ratios["P/E"]               = c["share_price"][idx] / eps
    ratios["P/B"]               = c["share_price"][idx] / bvps
    ratios["EV/EBITDA"]         = ev / c["ebitda"][idx]
    return ratios

# Print table
header_names = list(all_ratios_year(companies[0]).keys())
print(f"{'Ratio':<22}", end="")
for c in companies:
    print(f" {c['name']:>12}", end="")
print()
print("-" * 60)

for rname in header_names:
    print(f"{rname:<22}", end="")
    for c in companies:
        val = all_ratios_year(c)[rname]
        # Format percentages vs ratios
        if rname in ["Gross Margin", "Operating Margin", "Net Margin", "ROA", "ROE", "D/A"]:
            print(f" {val:>11.1%}", end="")
        else:
            print(f" {val:>11.2f}", end="")
    print()

### 11.2 Small-Multiples Visualization

Small-multiples plots show the **5-year trend** for several key ratios side by side, making it easy to spot diverging trends.

In [ ]:
# ---------------------------------------------------------------
# 11.2 Small-multiples: 5-year trends for key ratios
# ---------------------------------------------------------------

key_ratios_funcs = {
    "Net Margin (%)":       lambda c: c["net_income"] / c["revenue"] * 100,
    "ROE (%)":              lambda c: c["net_income"] / c["equity"] * 100,
    "Total Asset TO (x)":   lambda c: c["revenue"] / c["total_assets"],
    "Current Ratio (x)":    lambda c: c["current_assets"] / c["current_liab"],
    "D/E (x)":              lambda c: c["total_liab"] / c["equity"],
    "Interest Coverage (x)": lambda c: c["operating_income"] / c["interest_expense"],
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, (title, func) in zip(axes, key_ratios_funcs.items()):
    for c, col in zip(companies, colours):
        ax.plot(years, func(c), marker="o", color=col, linewidth=2, label=c["name"])
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks(years)
    ax.set_xlabel("Year")

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Multi-Company Dashboard — 5-Year Trends", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

### 11.3 DuPont Factor Decomposition — Stacked View

The chart below shows how each DuPont factor contributes to ROE across companies and years, providing a unified view of the decomposition dynamics.

> **CFA Exam Tip:** When asked to "explain the change in ROE," always decompose it.  Simply stating "ROE went up" without attribution to margin, turnover, or leverage will not earn full marks.

In [ ]:
# ---------------------------------------------------------------
# 11.3 DuPont 3-factor stacked bar chart (all companies, all years)
# ---------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, c, col in zip(axes, companies, [PRIMARY, SECONDARY, TERTIARY]):
    npm, ato, em, roe = dupont_3(c)

    # For stacked bar, show each factor's value (scaled for visibility)
    x = np.arange(n_years)
    w = 0.55

    ax.bar(x, npm * 100, w, label="Net Profit Margin (%)", color=PRIMARY, alpha=0.85)
    ax.bar(x, ato * 10, w, bottom=npm * 100, label="Asset Turnover (x10)", color=SECONDARY, alpha=0.85)
    ax.bar(x, em, w, bottom=npm * 100 + ato * 10, label="Equity Multiplier (x1)", color=TERTIARY, alpha=0.85)

    # Annotate ROE
    for i in range(n_years):
        total_h = npm[i] * 100 + ato[i] * 10 + em[i]
        ax.text(i, total_h + 0.3, f"ROE={roe[i]:.1%}", ha="center", fontsize=8, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([f"Yr {y}" for y in years])
    ax.set_title(c["name"], fontsize=12, fontweight="bold")

axes[0].set_ylabel("Factor Value (scaled)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("DuPont 3-Factor Components (Scaled for Visualisation)", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

## 12. Summary & Key Takeaways

> **Key Concept:** Ratio analysis transforms raw financial statements into comparable, interpretable metrics. The DuPont framework is the crown jewel — it tells you not just *what* ROE is, but *why* it is what it is.

**What we covered:**

1. **Five ratio families** — profitability, activity, liquidity, solvency, and valuation — each answering a different question about company health.
2. **DuPont 3-factor decomposition** — ROE = Net Margin x Asset Turnover x Leverage.
3. **DuPont 5-factor decomposition** — further separating tax burden, interest burden, and operating margin.
4. **Cross-sectional and time-series analysis** — two complementary lenses for interpreting ratios.
5. **Visualisation techniques** — radar charts, tornado charts, small-multiples dashboards.

> **CFA Exam Tip:** Be prepared to compute *and* interpret every ratio in this notebook.  The exam rarely asks for just a number — it asks what the number *means* for the company's financial health and what *actions* management could take to improve it.

---

## References

1. CFA Institute. *CFA Program Curriculum, Level I — Financial Statement Analysis.* CFA Institute, 2024.
2. Robinson, T. R., Henry, E., Pirie, W. L., & Broihahn, M. A. *International Financial Statement Analysis.* 4th ed., Wiley, 2020.
3. Penman, S. H. *Financial Statement Analysis and Security Valuation.* 6th ed., McGraw-Hill, 2022.
4. Palepu, K. G., Healy, P. M., & Peek, E. *Business Analysis and Valuation.* 6th ed., Cengage, 2019.
5. Damodaran, A. *Investment Valuation.* 3rd ed., Wiley, 2012.

---

*Notebook by Arjun — all code from scratch with NumPy and Matplotlib.*